Steps to be performed

1. import the library

2. load the documents

3. split the documents in the chunk

4. create the embeddings

5. store the embedding into chroma

6. create the retriever

7. initialize the language model

8. build the retrieved base QA chain

9. Ask a question

10. observe and explain the output

In [ ]:
!pip install langchain==0.1.16 langchain-community langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 28.7 MB/s eta 0:00:00


In [ ]:
pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found exis

In [ ]:
#import the library
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [ ]:
import os
#set up the api key
os.environ['OPENAI_API_KEY']='your-api-key'
#load the pdf file
pdf_path='/content/bhagavad-gita-in-english-source-file-2.pdf'
loader=PyPDFLoader(pdf_path)
documents=loader.load()


In [ ]:
#split the text into chunks
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks=splitter.split_documents(documents)

In [ ]:
#create the embedding and store in the chroma vector store
embeddings=OpenAIEmbeddings()
vector_store=Chroma.from_documents(chunks,embedding=embeddings)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(


In [ ]:
#create retrieval based QA chain
retriever=vector_store.as_retriever()
llm=ChatOpenAI(model_name='gpt-4o-mini',temperature=0)
qa_chain=RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    return_source_documents=True
)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [ ]:
#ask the question
query='who is Arjun?'
result=qa_chain(query)
#print the answer
print(result)

{'query': 'who is Arjun?', 'result': 'Arjuna is one of the five Pandava brothers in the Indian epic Mahabharata. He is a central character who faces a significant moral dilemma on the battlefield, where he must choose between fighting in a war against his own relatives and revered teacher or seeking peace. His struggles and decisions are explored in the context of the Bhagavad Gita, where he receives guidance from Lord Krishna.', 'source_documents': [Document(page_content='O Arjuna, know Me to be the eternal seed of all creatures. I am the \nintelligence of the intelligent and the brilliance of the brilliant. I am \nthe strength of the strong that is devoid of lust and attachment. I \nam lust (Cupid) in human beings that is in accord with righteous-\nness (Dharma) for the sacred and sole purpose of procreation after \nmarriage, O Arjuna. (7.10-11) Know that three modes (Gunas) of \nNature — goodness, passion, and ignorance — also emanate from', metadata={'page': 22, 'source': '/content

In [ ]:

#print the source
for doc in result['source_documents']:
  print(f'page content:{doc.page_content[:200]}')

page content:O Arjuna, know Me to be the eternal seed of all creatures. I am the 
intelligence of the intelligent and the brilliance of the brilliant. I am 
the strength of the strong that is devoid of lust and at
page content:posites born of likes and dislikes, O Arjuna. But persons of unself-
ish deeds, whose Karma or sin has come to an end, become free 
from the delusio n of pairs of opposites and worship Me with firm 
r
page content:olence. Arjuna, one of the five P andava brothers, faced th is di-
lemma in the battlefield: Whether to fight or run away from war for 
peace. 
 Arjuna’s dilemma is, in reality, the Universal Dilemma.
page content:Arjuna sat down on the back seat of the chariot with his mind 
overwhelmed with sorrow. (Verse 1.47) 
 
NOTE: Some non-essential verses from Chapters 1 and 2 only 
have been omitted for ease of unders
